
# URLFusionNet: Reproducible Experiment Notebook

**Paper:** *URLFusionNet: Adaptive Multi-Scale Character–Token Fusion for Multiclass Malicious URL Detection*

This notebook implements the experimental protocol defined in the manuscript:

- Kaggle **Malicious URLs Dataset** (`sid321axn/malicious-urls-dataset`)
- Four classes: **benign, defacement, phishing, malware**
- Dataset audit before splitting: missing values, duplicate URLs, label conflicts, URL validity
- Fixed stratified split: **80% train / 10% validation / 10% test**
- Training-only vocabulary adaptation
- Class-weighted optimization
- URLFusionNet with multi-scale character encoding, token self-attention, common projection, **interaction-aware sample-adaptive character–token fusion**, residual refinement, and four-class softmax output
- Primary selection metric: **validation Macro-F1**
- Final metrics: Accuracy, Macro-F1, Weighted-F1, MCC, class-wise Precision/Recall/F1, confusion matrix
- Baselines, ablation, robustness, multi-seed evaluation, model size, parameter count, and inference latency

## Runtime strategy

`EXPERIMENT_MODE = "QUICK"` verifies the pipeline quickly on the **full cleaned dataset**, but with fewer epochs and only seed 42.

For values reported in the paper, set:

```python
EXPERIMENT_MODE = "PAPER"
```

This activates seeds **42, 123, and 2025**. Do **not** report QUICK-mode numbers as final manuscript results.

Primary dataset:  
https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset


In [ ]:
# Optional installation cell for Google Colab / a fresh Jupyter environment.
import sys
import subprocess

for pkg in ["kagglehub", "xgboost"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


## 1. Imports, hardware configuration, and experiment profile

In [ ]:

import os
import gc
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SPLIT_SEED = 42

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(SPLIT_SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU(s):", gpus)

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled.")
    except Exception as e:
        print("Mixed precision not enabled:", e)

EXPERIMENT_MODE = "QUICK"   # "QUICK" or "PAPER"

if EXPERIMENT_MODE.upper() == "QUICK":
    SEEDS = [42]
    MAIN_EPOCHS = 6
    ABLATION_EPOCHS = 3
    BASELINE_EPOCHS = 4
    ML_MAX_ITER = 80
    ROBUSTNESS_N = 12000
else:
    SEEDS = [42, 123, 2025]
    MAIN_EPOCHS = 12
    ABLATION_EPOCHS = 6
    BASELINE_EPOCHS = 8
    ML_MAX_ITER = 150
    ROBUSTNESS_N = None

BATCH_SIZE = 1024 if gpus else 256
LEARNING_RATE = 2e-3 if gpus else 1e-3

MAX_CHAR_LEN = 200
MAX_TOKEN_LEN = 40
MAX_CHAR_VOCAB = 256
MAX_TOKEN_VOCAB = 30000
CHAR_EMB_DIM = 32
TOKEN_EMB_DIM = 48
CONV_FILTERS = 48
LATENT_DIM = 96
ATTN_HEADS = 2
DROPOUT = 0.20
NUM_CLASSES = 4

CLASS_NAMES = ["benign", "defacement", "phishing", "malware"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}

RESULT_DIR = Path("urlfusionnet_results")
RESULT_DIR.mkdir(exist_ok=True)

print({
    "mode": EXPERIMENT_MODE,
    "seeds": SEEDS,
    "batch_size": BATCH_SIZE,
    "main_epochs": MAIN_EPOCHS,
    "ablation_epochs": ABLATION_EPOCHS,
    "baseline_epochs": BASELINE_EPOCHS,
})


## 2. Download the Kaggle dataset and verify the expected corpus

In [ ]:

import glob

KAGGLE_DATASET = "sid321axn/malicious-urls-dataset"
EXPECTED_COUNTS = {
    "benign": 428103,
    "defacement": 96457,
    "phishing": 94111,
    "malware": 32520,
}

candidate_paths = [
    "/kaggle/input/malicious-urls-dataset/malicious_phish.csv",
    "./malicious_phish.csv",
]

csv_path = None
for p in candidate_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    import kagglehub
    data_root = kagglehub.dataset_download(KAGGLE_DATASET)
    matches = glob.glob(os.path.join(data_root, "**", "malicious_phish.csv"), recursive=True)
    if not matches:
        matches = glob.glob(os.path.join(data_root, "**", "*.csv"), recursive=True)
    if not matches:
        raise FileNotFoundError(
            "No CSV was found after Kaggle download. Set csv_path manually."
        )
    csv_path = matches[0]

print("Dataset path:", csv_path)
df_raw = pd.read_csv(csv_path)
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


In [ ]:

lower_cols = {c.lower().strip(): c for c in df_raw.columns}

url_col = next((lower_cols[k] for k in ["url", "urls", "link"] if k in lower_cols), None)
label_col = next((lower_cols[k] for k in ["type", "label", "class", "category"] if k in lower_cols), None)

if url_col is None or label_col is None:
    raise ValueError(f"Could not identify URL/label columns. Found: {df_raw.columns.tolist()}")

df = df_raw[[url_col, label_col]].copy()
df.columns = ["url", "label"]
df["url"] = df["url"].astype("string")
df["label"] = df["label"].astype("string").str.strip().str.lower()

original_counts = df["label"].value_counts()
display(original_counts.rename("count").to_frame())

observed = {k: int(original_counts.get(k, 0)) for k in EXPECTED_COUNTS}
print("Expected:", EXPECTED_COUNTS)
print("Observed:", observed)

if observed != EXPECTED_COUNTS:
    print(
        "WARNING: class counts differ from the manuscript's original Table II. "
        "Inspect the dataset version before reporting results."
    )
else:
    print("Dataset verification PASSED.")


## 3. Leakage-aware dataset audit

In [ ]:

audit = {}
audit["raw_rows"] = len(df)
audit["missing_url"] = int(df["url"].isna().sum())
audit["missing_label"] = int(df["label"].isna().sum())

df = df.dropna(subset=["url", "label"]).copy()
df["url"] = df["url"].str.strip()
df = df[df["url"].str.len() > 0].copy()

unknown_labels = sorted(set(df["label"].unique()) - set(CLASS_NAMES))
if unknown_labels:
    print("Unknown labels removed:", unknown_labels)
df = df[df["label"].isin(CLASS_NAMES)].copy()

valid_mask = df["url"].str.contains(r"\S", regex=True, na=False)
audit["invalid_url"] = int((~valid_mask).sum())
df = df[valid_mask].copy()

label_cardinality = df.groupby("url", sort=False)["label"].nunique()
conflict_urls = label_cardinality[label_cardinality > 1].index
audit["conflicting_unique_urls"] = int(len(conflict_urls))
audit["rows_in_label_conflicts"] = int(df["url"].isin(conflict_urls).sum())
df = df[~df["url"].isin(conflict_urls)].copy()

before_dedup = len(df)
df = df.drop_duplicates(subset=["url", "label"], keep="first").copy()
audit["same_label_duplicate_rows_removed"] = int(before_dedup - len(df))
audit["clean_rows"] = len(df)

audit_df = pd.DataFrame([audit])
display(audit_df)
audit_df.to_csv(RESULT_DIR / "dataset_audit.csv", index=False)

clean_counts = df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
display(clean_counts.rename("count").to_frame())


## 4. Fixed stratified 80:10:10 split and class weights

In [ ]:

df["label_id"] = df["label"].map(LABEL_TO_ID).astype("int32")

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SPLIT_SEED,
    stratify=df["label_id"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SPLIT_SEED,
    stratify=temp_df["label_id"],
)

for part in [train_df, val_df, test_df]:
    part.reset_index(drop=True, inplace=True)

split_rows = []
for split_name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    counts = part["label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
    row = {"split": split_name, "total": len(part)}
    row.update({c: int(counts[c]) for c in CLASS_NAMES})
    split_rows.append(row)

split_table = pd.DataFrame(split_rows)
display(split_table)
split_table.to_csv(RESULT_DIR / "dataset_split.csv", index=False)

train_counts = train_df["label_id"].value_counts().sort_index()
N_train = len(train_df)
CLASS_WEIGHT = {
    k: float(N_train / (NUM_CLASSES * train_counts.loc[k]))
    for k in range(NUM_CLASSES)
}
print("Class weights:", {ID_TO_LABEL[k]: round(v, 4) for k, v in CLASS_WEIGHT.items()})


## 5. Training-only character and token vectorizers

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
def url_token_standardize(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r"[^a-z0-9]+", " ")
    return tf.strings.strip(x)

char_vectorizer = layers.TextVectorization(
    max_tokens=MAX_CHAR_VOCAB,
    standardize=None,
    split="character",
    output_mode="int",
    output_sequence_length=MAX_CHAR_LEN,
    name="char_vectorizer",
)

token_vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKEN_VOCAB,
    standardize=url_token_standardize,
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_TOKEN_LEN,
    name="token_vectorizer",
)

adapt_ds = tf.data.Dataset.from_tensor_slices(
    train_df["url"].astype(str).values
).batch(4096)

print("Adapting character vocabulary...")
char_vectorizer.adapt(adapt_ds)

print("Adapting token vocabulary...")
token_vectorizer.adapt(adapt_ds)

print("Character vocabulary size:", len(char_vectorizer.get_vocabulary()))
print("Token vocabulary size:", len(token_vectorizer.get_vocabulary()))


## 6. TensorFlow input pipeline

In [ ]:

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, training=False, seed=42):
    urls = frame["url"].astype(str).values
    labels = frame["label_id"].astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((urls, labels))
    if training:
        ds = ds.shuffle(
            buffer_size=min(len(frame), 200_000),
            seed=seed,
            reshuffle_each_iteration=True,
        )
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def make_predict_dataset(urls):
    return (
        tf.data.Dataset.from_tensor_slices(np.asarray(urls, dtype=str))
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

train_ds = make_dataset(train_df, training=True, seed=SPLIT_SEED)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)
print("Datasets ready.")


## 7. URLFusionNet layers and Macro-F1 metric

In [ ]:

@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class PositionalEmbedding(layers.Layer):
    def __init__(self, max_length, dim, **kwargs):
        super().__init__(**kwargs)
        self.max_length = max_length
        self.dim = dim
        self.supports_masking = True

    def build(self, input_shape):
        self.positional = self.add_weight(
            name="positional_embedding",
            shape=(1, self.max_length, self.dim),
            initializer="random_normal",
            trainable=True,
        )

    def call(self, inputs):
        length = tf.shape(inputs)[1]
        return inputs + self.positional[:, :length, :]

    def compute_mask(self, inputs, mask=None):
        return mask

    def get_config(self):
        config = super().get_config()
        config.update({"max_length": self.max_length, "dim": self.dim})
        return config


@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AbsDifference(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.abs(a - b)


@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class AdaptiveWeightedFusion(layers.Layer):
    def call(self, inputs):
        zc, zt, alpha = inputs
        alpha_c = alpha[:, 0:1]
        alpha_t = alpha[:, 1:2]
        return alpha_c * zc + alpha_t * zt


@tf.keras.utils.register_keras_serializable(package="URLFusionNet")
class SparseMacroF1(tf.keras.metrics.Metric):
    def __init__(self, num_classes=4, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.cm = self.add_weight(
            name="confusion_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32,
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        cm_batch = tf.math.confusion_matrix(
            y_true, y_pred,
            num_classes=self.num_classes,
            dtype=tf.float32
        )
        self.cm.assign_add(cm_batch)

    def result(self):
        tp = tf.linalg.diag_part(self.cm)
        fp = tf.reduce_sum(self.cm, axis=0) - tp
        fn = tf.reduce_sum(self.cm, axis=1) - tp
        precision = tf.math.divide_no_nan(tp, tp + fp)
        recall = tf.math.divide_no_nan(tp, tp + fn)
        f1 = tf.math.divide_no_nan(2.0 * precision * recall, precision + recall)
        return tf.reduce_mean(f1)

    def reset_state(self):
        self.cm.assign(tf.zeros_like(self.cm))

    def get_config(self):
        config = super().get_config()
        config.update({"num_classes": self.num_classes})
        return config


## 8. URLFusionNet architecture and ablation variants

In [ ]:

def build_urlfusionnet(variant="full"):
    url_input = keras.Input(shape=(), dtype=tf.string, name="url")

    # Character branch: multi-scale Conv1D, k = 3, 5, 7
    char_ids = char_vectorizer(url_input)
    xc = layers.Embedding(
        input_dim=len(char_vectorizer.get_vocabulary()),
        output_dim=CHAR_EMB_DIM,
        name="char_embedding",
    )(char_ids)

    pooled_scales = []
    for k in (3, 5, 7):
        h = layers.Conv1D(
            CONV_FILTERS, kernel_size=k, padding="same",
            use_bias=False, name=f"char_conv_k{k}"
        )(xc)
        h = layers.BatchNormalization(name=f"char_bn_k{k}")(h)
        h = layers.Activation(tf.nn.gelu, name=f"char_gelu_k{k}")(h)
        h = layers.GlobalMaxPooling1D(name=f"char_gmp_k{k}")(h)
        pooled_scales.append(h)

    hc = layers.Concatenate(name="char_multiscale_concat")(pooled_scales)
    hc = layers.Dense(LATENT_DIM, name="char_scale_projection")(hc)

    if variant == "multiscale_char":
        r = layers.Dropout(DROPOUT)(hc)
        output = layers.Dense(
            NUM_CLASSES, activation="softmax", dtype="float32", name="classifier"
        )(r)
        return keras.Model(url_input, output, name="MultiScaleCharCNN")

    # Token self-attention branch
    token_ids = token_vectorizer(url_input)
    xt = layers.Embedding(
        input_dim=len(token_vectorizer.get_vocabulary()),
        output_dim=TOKEN_EMB_DIM,
        mask_zero=True,
        name="token_embedding",
    )(token_ids)
    xt = PositionalEmbedding(MAX_TOKEN_LEN, TOKEN_EMB_DIM, name="token_position")(xt)

    attn = layers.MultiHeadAttention(
        num_heads=ATTN_HEADS,
        key_dim=max(8, TOKEN_EMB_DIM // ATTN_HEADS),
        dropout=0.10,
        name="token_self_attention",
    )(xt, xt)
    xt = layers.LayerNormalization(name="token_attn_ln")(
        layers.Add()([xt, attn])
    )

    ff = layers.Dense(TOKEN_EMB_DIM * 2, activation=tf.nn.gelu, name="token_ff1")(xt)
    ff = layers.Dropout(DROPOUT)(ff)
    ff = layers.Dense(TOKEN_EMB_DIM, name="token_ff2")(ff)
    xt = layers.LayerNormalization(name="token_ff_ln")(
        layers.Add()([xt, ff])
    )
    ht = layers.GlobalAveragePooling1D(name="token_pool")(xt)

    # Common projection
    zc = layers.Dense(LATENT_DIM, name="char_common_projection")(hc)
    zc = layers.LayerNormalization(name="z_char")(zc)

    zt = layers.Dense(LATENT_DIM, name="token_common_projection")(ht)
    zt = layers.LayerNormalization(name="z_token")(zt)

    if variant == "char_only":
        fused = zc
    elif variant == "token_only":
        fused = zt
    elif variant == "fixed_avg":
        fused = layers.Average(name="fixed_average_fusion")([zc, zt])
    else:
        # Core novelty: interaction-aware sample-adaptive fusion
        if variant == "no_interaction":
            gate_input = layers.Concatenate(name="gate_no_interaction")([zc, zt])
        else:
            product = layers.Multiply(name="cross_granularity_product")([zc, zt])
            difference = AbsDifference(name="cross_granularity_absdiff")([zc, zt])
            gate_input = layers.Concatenate(name="interaction_representation")(
                [zc, zt, product, difference]
            )

        gate_hidden = layers.Dense(
            LATENT_DIM // 2, activation=tf.nn.gelu, name="gate_mlp"
        )(gate_input)
        alpha = layers.Dense(
            2, activation="softmax", dtype="float32", name="fusion_weights"
        )(gate_hidden)

        fused = AdaptiveWeightedFusion(name="adaptive_weighted_fusion")(
            [zc, zt, alpha]
        )

    # Residual refinement
    if variant != "no_residual" and variant not in ("char_only", "token_only"):
        residual = layers.Dense(
            LATENT_DIM * 2, activation=tf.nn.gelu, name="residual_fc1"
        )(fused)
        residual = layers.Dropout(DROPOUT)(residual)
        residual = layers.Dense(LATENT_DIM, name="residual_fc2")(residual)
        fused = layers.LayerNormalization(name="residual_ln")(
            layers.Add()([fused, residual])
        )

    output = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="classifier",
    )(fused)

    return keras.Model(url_input, output, name=f"URLFusionNet_{variant}")


def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,
            weight_decay=1e-4,
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            SparseMacroF1(NUM_CLASSES, name="macro_f1"),
        ],
    )
    return model

demo_model = compile_model(build_urlfusionnet("full"))
demo_model.summary(line_length=110)
del demo_model
gc.collect()


## 9. Evaluation and training helpers

In [ ]:

def predict_probabilities(model, frame):
    pred_ds = make_predict_dataset(frame["url"].astype(str).values)
    return model.predict(pred_ds, verbose=0)

def metric_row(y_true, y_pred, model_name, seed=None):
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "Model": model_name,
        "Seed": seed,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": p_macro,
        "Macro_Recall": r_macro,
        "Macro_F1": f_macro,
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

def evaluate_deep_model(model, frame, model_name, seed=None, save_report=True):
    y_true = frame["label_id"].to_numpy()
    probs = predict_probabilities(model, frame)
    y_pred = probs.argmax(axis=1)

    row = metric_row(y_true, y_pred, model_name, seed)

    if save_report:
        report = classification_report(
            y_true, y_pred,
            target_names=CLASS_NAMES,
            digits=6,
            output_dict=True,
            zero_division=0,
        )
        pd.DataFrame(report).T.to_csv(
            RESULT_DIR / f"classification_report_{model_name}_seed{seed}.csv"
        )
    return row, probs, y_pred

def plot_confusion(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()

def training_callbacks(tag):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=2,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_macro_f1",
            mode="max",
            factor=0.5,
            patience=1,
            min_lr=1e-5,
            verbose=1,
        ),
        keras.callbacks.CSVLogger(RESULT_DIR / f"history_{tag}.csv"),
    ]

def train_variant(variant, seed, epochs, verbose=1):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    ds = make_dataset(train_df, training=True, seed=seed)
    model = compile_model(build_urlfusionnet(variant))
    history = model.fit(
        ds,
        validation_data=val_ds,
        epochs=epochs,
        class_weight=CLASS_WEIGHT,
        callbacks=training_callbacks(f"{variant}_seed{seed}"),
        verbose=verbose,
    )
    return model, history


## 10. Fast traditional ML baselines

In [ ]:

def lexical_features(series: pd.Series) -> pd.DataFrame:
    s = series.fillna("").astype(str)
    length = s.str.len().clip(lower=1)

    X = pd.DataFrame(index=s.index)
    X["url_length"] = length
    X["dots"] = s.str.count(r"\.")
    X["slashes"] = s.str.count(r"/")
    X["hyphens"] = s.str.count(r"-")
    X["underscores"] = s.str.count(r"_")
    X["question_marks"] = s.str.count(r"\?")
    X["equals"] = s.str.count(r"=")
    X["ampersands"] = s.str.count(r"&")
    X["ats"] = s.str.count(r"@")
    X["percents"] = s.str.count(r"%")
    X["digits"] = s.str.count(r"\d")
    X["digit_ratio"] = X["digits"] / length
    X["https"] = s.str.lower().str.startswith("https://").astype("int8")
    X["has_ip_like"] = s.str.contains(
        r"(?:\d{1,3}\.){3}\d{1,3}", regex=True, na=False
    ).astype("int8")

    suspicious = ["login", "verify", "secure", "account", "update", "bank", "signin", "confirm"]
    score = pd.Series(0, index=s.index, dtype="int16")
    sl = s.str.lower()
    for word in suspicious:
        score += sl.str.contains(word, regex=False, na=False).astype("int16")
    X["suspicious_token_count"] = score
    return X.astype("float32")

ml_results = []

print("Extracting lexical baseline features...")
Xtr = lexical_features(train_df["url"])
Xte = lexical_features(test_df["url"])
ytr = train_df["label_id"].to_numpy()
yte = test_df["label_id"].to_numpy()

sgd = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=ML_MAX_ITER,
    tol=1e-3,
    random_state=42,
    n_jobs=-1,
)
t0 = time.perf_counter()
sgd.fit(Xtr, ytr)
pred = sgd.predict(Xte)
row = metric_row(yte, pred, "SGD-Logistic", 42)
row["Train_s"] = time.perf_counter() - t0
ml_results.append(row)

hgb = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=ML_MAX_ITER,
    max_leaf_nodes=31,
    early_stopping=True,
    random_state=42,
)
sw = np.asarray([CLASS_WEIGHT[int(y)] for y in ytr], dtype=np.float32)
t0 = time.perf_counter()
hgb.fit(Xtr, ytr, sample_weight=sw)
pred = hgb.predict(Xte)
row = metric_row(yte, pred, "HistGradientBoosting", 42)
row["Train_s"] = time.perf_counter() - t0
ml_results.append(row)

ml_results_df = pd.DataFrame(ml_results).sort_values("Macro_F1", ascending=False)
display(ml_results_df)
ml_results_df.to_csv(RESULT_DIR / "traditional_ml_baselines.csv", index=False)

del Xtr, Xte, ytr, yte, sw, sgd, hgb
gc.collect()



## 10B. Extended baselines aligned with Fig. 1

This section adds the baseline families explicitly shown in the research framework:

- **Linear SVM**
- **Random Forest**
- **XGBoost**
- **Char-CNN**
- **Token-BiLSTM**
- **CNN–Transformer Hybrid (TCURL-style)**

The **TCURL-style** model is a controlled reference implementation inspired by the local–global CNN/Transformer principle. It is **not presented as an exact reproduction of the published TCURL implementation**; therefore, report it in the paper as *CNN–Transformer Hybrid (TCURL-style)* unless the original implementation is reproduced exactly.


In [ ]:

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", e)

extended_ml_results = []

print("Preparing lexical feature matrix for SVM / RF / XGBoost ...")
Xtr_ext = lexical_features(train_df["url"])
Xte_ext = lexical_features(test_df["url"])
ytr_ext = train_df["label_id"].to_numpy()
yte_ext = test_df["label_id"].to_numpy()

# -------- Linear SVM --------
svm = LinearSVC(
    class_weight="balanced",
    max_iter=3000,
    random_state=42,
)
t0 = time.perf_counter()
svm.fit(Xtr_ext, ytr_ext)
pred = svm.predict(Xte_ext)
row = metric_row(yte_ext, pred, "LinearSVM", 42)
row["Train_s"] = time.perf_counter() - t0
extended_ml_results.append(row)
print("LinearSVM done.")

# -------- Random Forest --------
rf = RandomForestClassifier(
    n_estimators=160 if EXPERIMENT_MODE == "QUICK" else 250,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42,
)
t0 = time.perf_counter()
rf.fit(Xtr_ext, ytr_ext)
pred = rf.predict(Xte_ext)
row = metric_row(yte_ext, pred, "RandomForest", 42)
row["Train_s"] = time.perf_counter() - t0
extended_ml_results.append(row)
print("RandomForest done.")

# -------- XGBoost --------
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(
        n_estimators=220 if EXPERIMENT_MODE == "QUICK" else 350,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    sample_weight_xgb = np.asarray(
        [CLASS_WEIGHT[int(y)] for y in ytr_ext], dtype=np.float32
    )
    t0 = time.perf_counter()
    xgb.fit(Xtr_ext, ytr_ext, sample_weight=sample_weight_xgb)
    pred = xgb.predict(Xte_ext)
    row = metric_row(yte_ext, pred, "XGBoost", 42)
    row["Train_s"] = time.perf_counter() - t0
    extended_ml_results.append(row)
    print("XGBoost done.")

extended_ml_df = pd.DataFrame(extended_ml_results).sort_values(
    "Macro_F1", ascending=False
).reset_index(drop=True)

display(extended_ml_df)
extended_ml_df.to_csv(
    RESULT_DIR / "extended_traditional_baselines.csv",
    index=False
)

# Free memory before neural baselines.
del Xtr_ext, Xte_ext, ytr_ext, yte_ext, svm, rf
if XGBOOST_AVAILABLE:
    del xgb, sample_weight_xgb
gc.collect()


In [ ]:

def build_char_cnn():
    """Single-view character CNN baseline."""
    url_input = keras.Input(shape=(), dtype=tf.string, name="url")
    char_ids = char_vectorizer(url_input)
    x = layers.Embedding(
        len(char_vectorizer.get_vocabulary()),
        CHAR_EMB_DIM,
        name="char_embedding"
    )(char_ids)
    x = layers.Conv1D(64, 5, padding="same", activation=tf.nn.gelu)(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(96, 5, padding="same", activation=tf.nn.gelu)(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(
        NUM_CLASSES, activation="softmax", dtype="float32", name="classifier"
    )(x)
    return keras.Model(url_input, out, name="CharCNN")


def build_token_bilstm():
    """Single-view token BiLSTM baseline."""
    url_input = keras.Input(shape=(), dtype=tf.string, name="url")
    token_ids = token_vectorizer(url_input)
    x = layers.Embedding(
        len(token_vectorizer.get_vocabulary()),
        TOKEN_EMB_DIM,
        mask_zero=True,
        name="token_embedding"
    )(token_ids)
    x = layers.Bidirectional(
        layers.LSTM(48, return_sequences=False, dropout=0.10),
        name="bilstm"
    )(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(
        NUM_CLASSES, activation="softmax", dtype="float32", name="classifier"
    )(x)
    return keras.Model(url_input, out, name="TokenBiLSTM")


def build_tcurl_style():
    """
    Controlled CNN–Transformer hybrid reference.

    The character sequence feeds:
      (1) a CNN branch for local patterns;
      (2) a downsampled self-attention branch for global dependencies.

    The two branches are concatenated with fixed fusion.
    This is TCURL-style, not an exact reproduction of the original paper.
    """
    url_input = keras.Input(shape=(), dtype=tf.string, name="url")
    char_ids = char_vectorizer(url_input)

    x = layers.Embedding(
        len(char_vectorizer.get_vocabulary()),
        CHAR_EMB_DIM,
        name="char_embedding"
    )(char_ids)

    # Local CNN branch
    c = layers.Conv1D(
        64, 5, padding="same", activation=tf.nn.gelu, name="cnn_local"
    )(x)
    c = layers.GlobalMaxPooling1D(name="cnn_pool")(c)

    # Global Transformer-style branch.
    # Downsample first to keep memory/runtime reasonable on the full corpus.
    t = layers.AveragePooling1D(pool_size=4, name="transformer_downsample")(x)
    t = layers.Dense(48, name="transformer_projection")(t)
    t = PositionalEmbedding(
        MAX_CHAR_LEN // 4, 48, name="transformer_position"
    )(t)

    attn = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=24,
        dropout=0.10,
        name="transformer_attention"
    )(t, t)
    t = layers.LayerNormalization(name="transformer_ln1")(
        layers.Add()([t, attn])
    )

    ff = layers.Dense(96, activation=tf.nn.gelu, name="transformer_ff1")(t)
    ff = layers.Dense(48, name="transformer_ff2")(ff)
    t = layers.LayerNormalization(name="transformer_ln2")(
        layers.Add()([t, ff])
    )
    t = layers.GlobalAveragePooling1D(name="transformer_pool")(t)

    fused = layers.Concatenate(name="fixed_hybrid_concat")([c, t])
    fused = layers.Dense(96, activation=tf.nn.gelu)(fused)
    fused = layers.Dropout(DROPOUT)(fused)

    out = layers.Dense(
        NUM_CLASSES, activation="softmax", dtype="float32", name="classifier"
    )(fused)
    return keras.Model(url_input, out, name="CNNTransformer_TCURLStyle")


def train_external_baseline(builder_fn, model_name, seed=42, epochs=None):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    model = compile_model(builder_fn())
    ds = make_dataset(train_df, training=True, seed=seed)

    history = model.fit(
        ds,
        validation_data=val_ds,
        epochs=epochs or BASELINE_EPOCHS,
        class_weight=CLASS_WEIGHT,
        callbacks=training_callbacks(f"{model_name}_seed{seed}"),
        verbose=1,
    )

    row, probs, pred = evaluate_deep_model(
        model, test_df, model_name, seed=seed, save_report=True
    )
    return model, history, row, probs, pred


In [ ]:

extended_deep_results = []

DEEP_BASELINES = [
    (build_char_cnn, "CharCNN"),
    (build_token_bilstm, "TokenBiLSTM"),
    (build_tcurl_style, "CNNTransformer_TCURLStyle"),
]

for builder_fn, model_name in DEEP_BASELINES:
    print("\n" + "=" * 80)
    print("Training baseline:", model_name)
    print("=" * 80)

    model_b, history_b, row_b, probs_b, pred_b = train_external_baseline(
        builder_fn,
        model_name=model_name,
        seed=42,
        epochs=BASELINE_EPOCHS,
    )
    extended_deep_results.append(row_b)

    del model_b, history_b, probs_b, pred_b
    gc.collect()

extended_deep_df = pd.DataFrame(extended_deep_results).sort_values(
    "Macro_F1", ascending=False
).reset_index(drop=True)

display(extended_deep_df)
extended_deep_df.to_csv(
    RESULT_DIR / "extended_deep_baselines.csv",
    index=False
)


## 11. Multi-scale character deep baseline

In [ ]:

deep_baseline_results = []

baseline_model, _ = train_variant(
    variant="multiscale_char",
    seed=42,
    epochs=MAIN_EPOCHS if EXPERIMENT_MODE == "PAPER" else min(4, MAIN_EPOCHS),
)

row, _, baseline_pred = evaluate_deep_model(
    baseline_model, test_df, "MultiScaleCharCNN", seed=42
)
deep_baseline_results.append(row)

display(pd.DataFrame(deep_baseline_results))
pd.DataFrame(deep_baseline_results).to_csv(
    RESULT_DIR / "deep_baseline_results.csv", index=False
)

del baseline_model, baseline_pred
gc.collect()


## 12. Main URLFusionNet multi-seed experiment

In [ ]:

main_results = []
seed42_weights_path = RESULT_DIR / "URLFusionNet_seed42.weights.h5"

for seed in SEEDS:
    print("\n" + "=" * 80)
    print(f"Training full URLFusionNet | seed={seed}")
    print("=" * 80)

    model, history = train_variant(
        variant="full",
        seed=seed,
        epochs=MAIN_EPOCHS,
    )

    row, probs, pred = evaluate_deep_model(
        model, test_df, "URLFusionNet", seed=seed
    )
    main_results.append(row)

    if seed == 42:
        model.save_weights(seed42_weights_path)
        np.save(RESULT_DIR / "URLFusionNet_seed42_test_probabilities.npy", probs)
        np.save(RESULT_DIR / "URLFusionNet_seed42_test_predictions.npy", pred)

        plot_confusion(
            test_df["label_id"].to_numpy(),
            pred,
            "URLFusionNet Confusion Matrix (Seed 42)",
            "fig_confusion_matrix_seed42.png",
        )

    del model, history, probs, pred
    gc.collect()

main_results_df = pd.DataFrame(main_results)
display(main_results_df)
main_results_df.to_csv(RESULT_DIR / "urlfusionnet_multiseed_raw.csv", index=False)

summary = (
    main_results_df
    .drop(columns=["Seed"])
    .select_dtypes(include=np.number)
    .agg(["mean", "std"])
    .T
)
display(summary)
summary.to_csv(RESULT_DIR / "urlfusionnet_multiseed_mean_std.csv")


## 13. Reload seed-42 model for controlled analyses

In [ ]:
# Rebuild the architecture and load seed-42 weights.
# This avoids Windows cp1252/.keras serialization problems with custom objects.
tf.keras.backend.clear_session()
set_all_seeds(42)

main_model = compile_model(build_urlfusionnet("full"))
main_model.load_weights(seed42_weights_path)

print("Seed-42 URLFusionNet weights reloaded successfully.")


## 14. Ablation study

In [ ]:

ABLATION_VARIANTS = [
    ("char_only", "Character-only"),
    ("token_only", "Token-only"),
    ("fixed_avg", "FixedAvg"),
    ("no_interaction", "NoInteraction"),
    ("no_residual", "NoResidual"),
]

ablation_results = []

for variant, display_name in ABLATION_VARIANTS:
    print("\nAblation:", display_name)
    model, _ = train_variant(
        variant=variant,
        seed=42,
        epochs=ABLATION_EPOCHS,
    )
    row, _, _ = evaluate_deep_model(
        model, test_df, display_name, seed=42, save_report=False
    )
    ablation_results.append(row)
    del model
    gc.collect()

full_seed42 = main_results_df[main_results_df["Seed"] == 42].iloc[0].to_dict()
full_seed42["Model"] = "URLFusionNet (Full)"
ablation_results.append(full_seed42)

ablation_df = pd.DataFrame(ablation_results).sort_values(
    "Macro_F1", ascending=False
).reset_index(drop=True)
display(ablation_df)
ablation_df.to_csv(RESULT_DIR / "ablation_results.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(ablation_df["Model"], ablation_df["Macro_F1"])
ax.set_ylabel("Macro-F1")
ax.set_title("URLFusionNet Ablation Study")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(RESULT_DIR / "fig_ablation_macro_f1.png", dpi=300, bbox_inches="tight")
plt.show()



## 15. Adaptive-gate analysis

This cell tests whether the learned fusion weights \(\alpha_i^c\) and \(\alpha_i^t\) actually vary across samples/classes rather than collapsing to a fixed weighting.


In [ ]:

gate_extractor = keras.Model(
    inputs=main_model.input,
    outputs=main_model.get_layer("fusion_weights").output,
)

gate_weights = gate_extractor.predict(
    make_predict_dataset(test_df["url"].astype(str).values),
    verbose=0,
)

gate_df = pd.DataFrame({
    "label": test_df["label"].values,
    "alpha_character": gate_weights[:, 0],
    "alpha_token": gate_weights[:, 1],
})

gate_summary = gate_df.groupby("label")[["alpha_character", "alpha_token"]].agg(
    ["mean", "std"]
)
display(gate_summary)
gate_df.to_csv(RESULT_DIR / "adaptive_gate_weights_test.csv", index=False)
gate_summary.to_csv(RESULT_DIR / "adaptive_gate_summary_by_class.csv")

gate_means = gate_df.groupby("label")[["alpha_character", "alpha_token"]].mean()
ax = gate_means.plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("Mean adaptive weight")
ax.set_title("Sample-Adaptive Fusion Weights by URL Class")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_DIR / "fig_adaptive_gate_weights.png", dpi=300, bbox_inches="tight")
plt.show()

print("Std(alpha_character):", float(gate_df["alpha_character"].std()))
print("Std(alpha_token):", float(gate_df["alpha_token"].std()))


## 16. Robustness analysis

In [ ]:

def stratified_subset(frame, n, seed=42):
    if n is None or n >= len(frame):
        return frame.copy().reset_index(drop=True)
    frac = n / len(frame)
    pieces = []
    for _, group in frame.groupby("label_id"):
        take = max(1, int(round(len(group) * frac)))
        pieces.append(group.sample(n=min(take, len(group)), random_state=seed))
    out = pd.concat(pieces, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)
    return out.reset_index(drop=True)

robust_df = stratified_subset(test_df, ROBUSTNESS_N, seed=42)
print("Robustness evaluation URLs:", len(robust_df))

LOOKALIKE = {
    "a": "4", "e": "3", "i": "1", "o": "0", "s": "5",
    "A": "4", "E": "3", "I": "1", "O": "0", "S": "5",
}

def _rng_for(text, salt):
    import hashlib
    h = hashlib.md5((salt + "||" + text).encode("utf-8", errors="ignore")).hexdigest()
    return random.Random(int(h[:8], 16))

def perturb_insert(url):
    url = str(url)
    if not url:
        return url
    rng = _rng_for(url, "insert")
    pos = rng.randrange(1, max(2, len(url)))
    return url[:pos] + rng.choice("abcdefghijklmnopqrstuvwxyz0123456789") + url[pos:]

def perturb_delete(url):
    url = str(url)
    if len(url) < 4:
        return url
    rng = _rng_for(url, "delete")
    candidates = [i for i, ch in enumerate(url) if ch.isalnum()]
    if not candidates:
        return url
    pos = rng.choice(candidates)
    return url[:pos] + url[pos+1:]

def perturb_substitute(url):
    url = str(url)
    candidates = [i for i, ch in enumerate(url) if ch in LOOKALIKE]
    if not candidates:
        return perturb_insert(url)
    rng = _rng_for(url, "substitute")
    pos = rng.choice(candidates)
    return url[:pos] + LOOKALIKE[url[pos]] + url[pos+1:]

def perturb_path(url):
    url = str(url)
    marker = "/verify-update"
    if "?" in url:
        base, query = url.split("?", 1)
        return base.rstrip("/") + marker + "?" + query
    return url.rstrip("/") + marker

def perturb_query(url):
    url = str(url)
    joiner = "&" if "?" in url else "?"
    return url + joiner + "ref=secure_update"

PERTURBATIONS = {
    "Clean": lambda x: x,
    "Character insertion": perturb_insert,
    "Character deletion": perturb_delete,
    "Character substitution": perturb_substitute,
    "Path modification": perturb_path,
    "Query manipulation": perturb_query,
}

robustness_rows = []
y_true_robust = robust_df["label_id"].to_numpy()

for name, fn in PERTURBATIONS.items():
    print("Evaluating:", name)
    perturbed_urls = robust_df["url"].map(fn)
    probs = main_model.predict(make_predict_dataset(perturbed_urls.values), verbose=0)
    pred = probs.argmax(axis=1)
    robustness_rows.append(metric_row(
        y_true_robust, pred, name, seed=42
    ))

robustness_results = pd.DataFrame(robustness_rows)
display(robustness_results)
robustness_results.to_csv(RESULT_DIR / "robustness_results.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(robustness_results["Model"], robustness_results["Macro_F1"])
ax.set_ylabel("Macro-F1")
ax.set_title("Robustness Under Controlled URL Perturbations")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(RESULT_DIR / "fig_robustness_macro_f1.png", dpi=300, bbox_inches="tight")
plt.show()


## 17. Computational efficiency

In [ ]:

def measure_efficiency(model, urls, warmup=2):
    """
    Measure deployment efficiency without serializing the full Keras model.
    Model size is computed directly from learned weight arrays.
    """
    urls = np.asarray(urls, dtype=str)
    ds = make_predict_dataset(urls)

    # Warm-up so graph tracing / GPU initialization does not dominate latency.
    for _ in range(warmup):
        _ = model.predict(ds.take(2), verbose=0)

    # Timed inference.
    t0 = time.perf_counter()
    _ = model.predict(ds, verbose=0)
    elapsed = time.perf_counter() - t0

    # Weight footprint in MB; avoids model.save(.keras) and Windows encoding issues.
    model_size_mb = (
        sum(np.asarray(w).nbytes for w in model.get_weights())
        / (1024 ** 2)
    )

    return {
        "Parameters": int(model.count_params()),
        "Model_Size_MB": float(model_size_mb),
        "Inference_ms_per_URL": float(1000.0 * elapsed / len(urls)),
        "Throughput_URL_per_s": float(len(urls) / elapsed),
        "Total_Inference_s": float(elapsed),
        "Latency_Sample_N": int(len(urls)),
    }


latency_sample = test_df["url"].iloc[: min(20000, len(test_df))].values

efficiency = measure_efficiency(
    main_model,
    latency_sample,
    warmup=2,
)

efficiency_df = pd.DataFrame([
    {"Model": "URLFusionNet", **efficiency}
])

display(efficiency_df)

efficiency_df.to_csv(
    RESULT_DIR / "efficiency_results.csv",
    index=False
)

print(
    f"Parameters           : {efficiency['Parameters']:,}\n"
    f"Model size (weights) : {efficiency['Model_Size_MB']:.3f} MB\n"
    f"Latency              : {efficiency['Inference_ms_per_URL']:.4f} ms/URL\n"
    f"Throughput           : {efficiency['Throughput_URL_per_s']:.2f} URLs/s"
)


## 18. Combined baseline comparison

In [ ]:

combined_rows = []

# Existing compact ML baselines
combined_rows.extend(ml_results)

# Extended traditional baselines aligned with Fig. 1
combined_rows.extend(extended_ml_results)

# Existing multi-scale character baseline
combined_rows.extend(deep_baseline_results)

# Extended deep / hybrid baselines
combined_rows.extend(extended_deep_results)

# Full proposed URLFusionNet, seed 42, for controlled comparison
combined_rows.append(
    main_results_df[main_results_df["Seed"] == 42].iloc[0].to_dict()
)

comparison_df = pd.DataFrame(combined_rows).sort_values(
    "Macro_F1", ascending=False
).reset_index(drop=True)

display(comparison_df)
comparison_df.to_csv(
    RESULT_DIR / "baseline_comparison.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(comparison_df["Model"], comparison_df["Macro_F1"])
ax.set_ylabel("Macro-F1")
ax.set_title("Baseline Comparison vs URLFusionNet")
ax.tick_params(axis="x", rotation=40)
fig.tight_layout()
fig.savefig(
    RESULT_DIR / "fig_baseline_comparison_macro_f1.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 19. Method consistency checks and final exports

In [ ]:

checks = {
    "Primary Kaggle corpus matches original class counts":
        observed == EXPECTED_COUNTS,
    "Split approximately 80/10/10":
        abs(len(train_df)/len(df) - 0.80) < 0.002
        and abs(len(val_df)/len(df) - 0.10) < 0.002
        and abs(len(test_df)/len(df) - 0.10) < 0.002,
    "No URL overlap train/validation":
        len(set(train_df.url).intersection(set(val_df.url))) == 0,
    "No URL overlap train/test":
        len(set(train_df.url).intersection(set(test_df.url))) == 0,
    "No URL overlap validation/test":
        len(set(val_df.url).intersection(set(test_df.url))) == 0,
    "PAPER mode uses manuscript seeds 42,123,2025":
        (EXPERIMENT_MODE != "PAPER") or (SEEDS == [42, 123, 2025]),
}

check_df = pd.DataFrame(
    [{"Check": k, "Passed": bool(v)} for k, v in checks.items()]
)
display(check_df)
check_df.to_csv(RESULT_DIR / "method_consistency_checks.csv", index=False)

if EXPERIMENT_MODE == "QUICK":
    print(
        "\nQUICK MODE COMPLETED.\n"
        "Use these outputs only to verify the pipeline. "
        "Set EXPERIMENT_MODE='PAPER' and rerun all cells before reporting final paper values."
    )
else:
    print(
        "\nPAPER MODE COMPLETED.\n"
        "Multi-seed, ablation, robustness, gate analysis, confusion matrix, "
        "and efficiency outputs are ready for the Results section."
    )

print("\nSaved outputs:")
for p in sorted(RESULT_DIR.glob("*")):
    print(" -", p)



## 20. Output-to-paper map

| Manuscript requirement | Output |
|---|---|
| Dataset audit / leakage control | `dataset_audit.csv`, `method_consistency_checks.csv` |
| Fixed 80:10:10 split | `dataset_split.csv` |
| Traditional ML baselines | `traditional_ml_baselines.csv`, `extended_traditional_baselines.csv` |
| Deep / hybrid baselines | `deep_baseline_results.csv`, `extended_deep_baselines.csv` |
| URLFusionNet per-seed results | `urlfusionnet_multiseed_raw.csv` |
| Mean ± SD | `urlfusionnet_multiseed_mean_std.csv` |
| Per-class Precision/Recall/F1 | `classification_report_URLFusionNet_seed*.csv` |
| Confusion matrix | `fig_confusion_matrix_seed42.png` |
| Character-only / token-only / fixed fusion / removals | `ablation_results.csv` |
| Adaptive-gate evidence | `adaptive_gate_summary_by_class.csv`, `fig_adaptive_gate_weights.png` |
| Perturbation robustness | `robustness_results.csv`, `fig_robustness_macro_f1.png` |
| Parameters / size / latency | `efficiency_results.csv` |
| Overall comparison | `baseline_comparison.csv`, `fig_baseline_comparison_macro_f1.png` |

### Novelty interpretation rule

The adaptive fusion claim is supported only if the full model is competitive or superior on **Macro-F1**, improves over fixed/single-view variants, produces non-degenerate per-sample fusion weights, remains reasonably robust under perturbations, and retains practical computational cost. Do not claim superiority if the measured results do not support it.
